# E2 · A tree with AUC 0.99 on loans: it was the identifier

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Find out why a tree on the loans reaches AUC 0.99 in cross-validation: the dataset includes `customer_id`, assigned in order of **file closing** (defaulted loans have higher ids), so a threshold on the id separates almost perfectly. Measure the AUC on **new customers** (ids even higher) and
remove the column.

## Dataset

`dataset()` (given): 1500 training customers from `ds.loans(with_missing=False)` with `ratio` and a column `customer_id` assigned in order of outcome (first the regular ones, then the defaulters); `new` is a group of 300 customers with **consecutive ids above the maximum** of the training set (as would happen in production). Tree: `DecisionTreeClassifier(min_samples_leaf=30, random_state=0)`.

## Educational purpose

Recognize identifier leakage: any column that encodes **when or how** the data was collected (progressive ids, timestamps, file names, insertion order) can correlate with the target in a non-reproducible way. A tree, which looks for thresholds on any column, finds it right away.

## Task

1. Run the colleague's cell: `cv_auc_bug` (cross-validation AUC with `customer_id`) and `id_importance` (share of `feature_importances_` of the tree trained on all the data).
2. Compute `id_outcome_corr` (Spearman correlation between `customer_id` and `defaulted`) and `new_auc_bug`: AUC on `X_new`, `y_new` with the model trained with the id on the full training set.
3. Fix it: remove `customer_id` and save `cv_auc_ok` and `new_auc_ok`.
4. Save `n_distinct_new_values` (number of distinct values of `predict_proba` of the model with the id on the new customers): what does the model see?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier


def dataset():
    df = ds.loans(n=1800, with_missing=False)
    df["ratio"] = df["amount"] / df["income"]
    df = df.sample(frac=1, random_state=0).reset_index(drop=True)
    train, new = df.iloc[:1500].copy(), df.iloc[1500:].copy()
    train = train.sort_values("defaulted", kind="stable").reset_index(drop=True)      # ids in order of outcome: first the regular ones, then the defaulters
    train["customer_id"] = np.arange(1500)
    new["customer_id"] = np.arange(1500, 1800)                                         # the new customers have ids above the maximum of the training set
    return train, new


train, new = dataset()
FEATURES = ["ratio", "age", "job_tenure", "income", "amount"]
X_tr, y_tr = train[FEATURES + ["customer_id"]], train["defaulted"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("defaulters in the training set:", round(y_tr.mean(), 3), "| ids of the new customers:", int(new["customer_id"].min()), "-", int(new["customer_id"].max()))

## Your work

### The colleague's code

In [ ]:
tree = DecisionTreeClassifier(min_samples_leaf=30, random_state=0)
cv_auc_bug = cross_val_score(tree, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
id_importance = pd.Series(tree.fit(X_tr, y_tr).feature_importances_, index=X_tr.columns)["customer_id"]
print(f"cross-validation AUC: {cv_auc_bug:.3f} | importance of customer_id: {id_importance:.2f}")

### Your diagnosis and the fix

In [ ]:
id_outcome_corr = None
new_auc_bug = None
n_distinct_new_values = None

In [ ]:
cv_auc_ok = None
new_auc_ok = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='New customers: `X_new = new[FEATURES + ["customer_id"]]`, `y_new = new["defaulted"]`; `roc_auc_score(y_new, model.predict_proba(X_new)[:, 1])` with `model = DecisionTreeClassifier(...).fit(X_tr, y_tr)`. Without the id: `X_tr[FEATURES]` and `X_new[FEATURES]`.'):
    check(cv_auc_bug > 0.95 and id_importance > 0.8, "the colleague: AUC over 0.95 and customer_id is worth over 80% of the importance", "don't modify the colleague's code")
    check(id_outcome_corr is not None and id_outcome_corr > 0.6, "the id is strongly correlated with the outcome (Spearman over 0.6)", "spearmanr(train['customer_id'], train['defaulted'])")
    check(new_auc_bug is not None and 0.45 < new_auc_bug < 0.55 and n_distinct_new_values <= 2, "on the new customers the AUC is 0.5: the model with the id gives everyone the same score (at most two distinct values)", "all the new ids are beyond the last threshold")
    check(cv_auc_ok is not None and 0.65 < cv_auc_ok < 0.85 and new_auc_ok > 0.62, "without the id: cross-validation AUC 0.65-0.85, on the new customers over 0.62", "FEATURES only (without customer_id)")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_the_customer_id_that_predicts_the_outcome.ipynb`. Open it after you pass the check, or after you have really tried.